In [ ]:
"""
LLM增强遗传编程因子挖掘系统 - 改进版
改进内容：
1. 增加种群大小和代数
2. 增加树深度
3. 添加更多金融基因（momentum20, macd, bbands等）
4. 训练/验证/测试三阶段分割（防止过拟合）
5. 因子相关性筛选
6. 早停机制
7. 结果保存
"""

import numpy as np
import pandas as pd
from deap import base, creator, tools, gp, algorithms
import operator
import random
import warnings
from scipy.stats import spearmanr
from functools import partial
import pickle
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# ==================== 1. 数据加载 ====================

def parse_dates(df, date_col='trade_date'):
    sample = str(df[date_col].iloc[0])
    print(f"检测到日期格式: {sample}")
    
    formats = ['%Y-%m-%d', '%Y%m%d', '%Y/%m/%d', '%d-%m-%Y', '%d/%m/%Y', 'mixed']
    
    for fmt in formats:
        try:
            if fmt == 'mixed':
                df[date_col] = pd.to_datetime(df[date_col])
            else:
                df[date_col] = pd.to_datetime(df[date_col], format=fmt)
            print(f"✅ 使用格式: {fmt}")
            return df
        except:
            continue
    
    df[date_col] = pd.to_datetime(df[date_col])
    return df

def load_data(filepath):
    print("="*60)
    print("📂 加载数据")
    print("="*60)
    
    df = pd.read_csv(filepath)
    print(f"原始数据: {len(df)}行, {len(df.columns)}列")
    
    df = parse_dates(df, 'trade_date')
    df = df.sort_values(['trade_date', 'ts_code']).reset_index(drop=True)
    
    df['future_ret'] = df.groupby('ts_code')['close'].pct_change().shift(-1)
    df = df.dropna(subset=['future_ret', 'close', 'vol'])
    
    stocks = df['ts_code'].nunique()
    days = df['trade_date'].nunique()
    print(f"✅ {days}天 × {stocks}只股票 = {len(df)}条记录")
    
    return df

# ==================== 2. 金融基因====================

def to_float_array(x, n):
    if x is None:
        return np.zeros(n, dtype=np.float64)
    if isinstance(x, pd.Series):
        x = x.values
    if isinstance(x, np.ndarray):
        if x.dtype.type is np.str_ or x.dtype.type is np.object_:
            return np.zeros(n, dtype=np.float64)
        if np.issubdtype(x.dtype, np.datetime64):
            return np.zeros(n, dtype=np.float64)
        try:
            return x.astype(np.float64)
        except:
            return np.zeros(n, dtype=np.float64)
    if isinstance(x, (int, float)):
        return np.full(n, float(x), dtype=np.float64)
    try:
        return np.array(x, dtype=np.float64)
    except:
        return np.zeros(n, dtype=np.float64)

# --- 原有基因 ---
def gene_gap(df):
    result = df.groupby('ts_code')['open'].transform(lambda x: x / x.shift(1)) - 1
    return to_float_array(result, len(df))

def gene_vol_ratio(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(20, min_periods=10).mean()
    )
    return to_float_array(result, len(df))

def gene_amplitude(df):
    result = (df['high'] - df['low']) / df.groupby('ts_code')['close'].transform(lambda x: x.shift(1))
    return to_float_array(result, len(df))

def gene_upper_shadow(df):
    amp = df['high'] - df['low']
    result = (df['high'] - df['close']) / amp.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_lower_shadow(df):
    amp = df['high'] - df['low']
    result = (df['open'] - df['low']) / amp.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_momentum5(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(5) - 1)
    return to_float_array(result, len(df))

def gene_momentum10(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(10) - 1)
    return to_float_array(result, len(df))

def gene_rsi14(df):
    def _calc(group):
        diff = group.diff()
        gain = diff.clip(lower=0)
        loss = (-diff).clip(lower=0)
        avg_gain = gain.rolling(14, min_periods=7).mean()
        avg_loss = loss.rolling(14, min_periods=7).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_vol_price(df):
    price_pct = df.groupby('ts_code')['close'].transform(lambda x: x.pct_change())
    vol_pct = df.groupby('ts_code')['vol'].transform(lambda x: x.pct_change())
    result = price_pct / vol_pct.replace(0, np.nan)
    return to_float_array(result, len(df))

# --- 新增基因 ---
def gene_momentum20(df):
    """20日动量"""
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(20) - 1)
    return to_float_array(result, len(df))

def gene_macd(df):
    """MACD：快线-慢线"""
    def _calc(group):
        ema12 = group.ewm(span=12, adjust=False).mean()
        ema26 = group.ewm(span=26, adjust=False).mean()
        return ema12 - ema26
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_bbands_width(df):
    """布林带宽度：(上轨-下轨)/中轨"""
    def _calc(group):
        ma = group.rolling(20, min_periods=10).mean()
        std = group.rolling(20, min_periods=10).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (upper - lower) / ma.replace(0, np.nan)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_atr(df):
    """ATR：平均真实波幅"""
    def _calc(group):
        high = group  # 这里需要用实际的高低价，但transform只能处理单列
        return group.rolling(14, min_periods=7).std()
    # 简化版本：使用波动率近似
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x.pct_change().rolling(14, min_periods=7).std()
    )
    return to_float_array(result, len(df))

def gene_volume_ma_ratio(df):
    """成交量与5日均量比"""
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(5, min_periods=3).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma(df):
    """价格与20日均线比"""
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(20, min_periods=10).mean()
    )
    return to_float_array(result, len(df))

def gene_volume_price_corr(df):
    """量价相关系数（滚动10日）"""
    def _calc(group):
        # 计算价格变化和成交量的相关系数
        price_ret = group.pct_change()
        # 这里简化，使用transform无法直接处理多列
        return price_ret.rolling(10, min_periods=5).std()
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

# 基因字典
GENES = {
    'gap': gene_gap,
    'vol_ratio': gene_vol_ratio,
    'amplitude': gene_amplitude,
    'upper_shadow': gene_upper_shadow,
    'lower_shadow': gene_lower_shadow,
    'momentum5': gene_momentum5,
    'momentum10': gene_momentum10,
    'momentum20': gene_momentum20,
    'rsi': gene_rsi14,
    'vol_price': gene_vol_price,
    'macd': gene_macd,
    'bbands_width': gene_bbands_width,
    'atr': gene_atr,
    'vol_ma_ratio': gene_volume_ma_ratio,
    'price_to_ma': gene_price_to_ma,
}
print(f"✅ 加载 {len(GENES)} 个金融基因")

# ==================== 3. GP定义 ====================

def safe_div(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(np.abs(y) > 1e-10, x / y, 1.0)
        res = np.where(np.isinf(res), 1.0, res)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_add(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x + y

def safe_sub(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x - y

def safe_mul(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x * y

def safe_log(x):
    x = np.array(x, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(x > 1e-10, np.log(np.abs(x)), 0.0)
        res = np.where(np.isnan(res), 0.0, res)
        res = np.where(np.isinf(res), 0.0, res)
    return res

def safe_abs(x):
    x = np.array(x, dtype=np.float64)
    return np.abs(x)

def safe_sqrt(x):
    x = np.array(x, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(x >= 0, np.sqrt(x), 0.0)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_pow(x, y):
    """安全幂运算"""
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where((x >= 0) | (np.abs(y) < 1e-10), np.power(np.abs(x), y), 0.0)
        res = np.where(np.isinf(res), 0.0, res)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def rand_int():
    return random.choice([3, 5, 10, 15, 20, 30])

def rand_float():
    return random.uniform(0.5, 2.0)

# 创建原始集
pset = gp.PrimitiveSetTyped("MAIN", [], np.ndarray)

# 算子
pset.addPrimitive(safe_div, [np.ndarray, np.ndarray], np.ndarray, name="div")
pset.addPrimitive(safe_add, [np.ndarray, np.ndarray], np.ndarray, name="add")
pset.addPrimitive(safe_sub, [np.ndarray, np.ndarray], np.ndarray, name="sub")
pset.addPrimitive(safe_mul, [np.ndarray, np.ndarray], np.ndarray, name="mul")
pset.addPrimitive(safe_log, [np.ndarray], np.ndarray, name="log")
pset.addPrimitive(safe_abs, [np.ndarray], np.ndarray, name="abs")
pset.addPrimitive(safe_sqrt, [np.ndarray], np.ndarray, name="sqrt")
pset.addPrimitive(safe_pow, [np.ndarray, np.ndarray], np.ndarray, name="pow")

# 终端
pset.addTerminal('close', np.ndarray)
pset.addTerminal('open', np.ndarray)
pset.addTerminal('high', np.ndarray)
pset.addTerminal('low', np.ndarray)
pset.addTerminal('volume', np.ndarray)

for name in GENES.keys():
    pset.addTerminal(name, np.ndarray)

pset.addEphemeralConstant("rand_int", rand_int, int)
pset.addEphemeralConstant("rand_float", rand_float, float)

print("✅ GP原始集配置完成")

# ==================== 4. 因子引擎 ====================

class FastFactorEngine:
    def __init__(self, df, name="Engine"):
        self.name = name
        self.df = df
        self.n = len(df)
        self.cache = {}
        
        self.cache.update({
            'close': df['close'].values.astype(np.float64),
            'open': df['open'].values.astype(np.float64),
            'high': df['high'].values.astype(np.float64),
            'low': df['low'].values.astype(np.float64),
            'volume': df['vol'].values.astype(np.float64),
        })
        
        print(f"计算金融基因 ({name})...")
        for i, (name_gene, func) in enumerate(GENES.items(), 1):
            try:
                result = func(df)
                self.cache[name_gene] = np.array(result, dtype=np.float64)
                print(f"  [{i}/{len(GENES)}] ✓ {name_gene}")
            except Exception as e:
                print(f"  [{i}/{len(GENES)}] ✗ {name_gene}: {e}")
                self.cache[name_gene] = np.zeros(self.n, dtype=np.float64)
        
        self.dates = df['trade_date'].values
        self.codes = df['ts_code'].values
        self.future_ret = df['future_ret'].values.astype(np.float64)
        
        self.date_to_indices = {}
        for idx, date in enumerate(self.dates):
            self.date_to_indices.setdefault(date, []).append(idx)
        
        self.unique_dates = sorted(self.date_to_indices.keys())
        print(f"✅ {name} 引擎就绪: {len(self.unique_dates)}天, {self.n}条记录")
    
    def evaluate_tree(self, expr_tree):
        """评估表达式树"""
        try:
            if isinstance(expr_tree, gp.Primitive):
                return self._eval_primitive(expr_tree)
            elif isinstance(expr_tree, gp.Terminal):
                return self._eval_terminal(expr_tree)
            elif isinstance(expr_tree, list):
                return self._eval_list(expr_tree)
            else:
                return np.zeros(self.n, dtype=np.float64)
        except Exception as e:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_primitive(self, node):
        try:
            name = node.name
            args = [self.evaluate_tree(arg) for arg in node.args]
            
            args_clean = []
            for a in args:
                if isinstance(a, np.ndarray):
                    args_clean.append(a.astype(np.float64))
                else:
                    args_clean.append(np.array(a, dtype=np.float64))
            args = args_clean
            
            if name == 'add':
                return safe_add(args[0], args[1])
            elif name == 'sub':
                return safe_sub(args[0], args[1])
            elif name == 'mul':
                return safe_mul(args[0], args[1])
            elif name == 'div':
                return safe_div(args[0], args[1])
            elif name == 'log':
                return safe_log(args[0])
            elif name == 'abs':
                return safe_abs(args[0])
            elif name == 'sqrt':
                return safe_sqrt(args[0])
            elif name == 'pow':
                return safe_pow(args[0], args[1])
            else:
                return args[0] if args else np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_terminal(self, node):
        try:
            val = node.value
            if isinstance(val, str):
                if val in self.cache:
                    return self.cache[val].astype(np.float64)
                else:
                    return np.zeros(self.n, dtype=np.float64)
            elif isinstance(val, (int, float)):
                return np.full(self.n, float(val), dtype=np.float64)
            else:
                return np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_list(self, expr):
        try:
            if len(expr) >= 1 and callable(expr[0]):
                args = [self.evaluate_tree(arg) for arg in expr[1:]]
                args = [np.array(a, dtype=np.float64) if not isinstance(a, np.ndarray) else a for a in args]
                result = expr[0](*args)
                return np.array(result, dtype=np.float64)
            return self.evaluate_tree(expr[0]) if expr else np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def calc_ic(self, factor_values):
        try:
            factor_values = np.array(factor_values, dtype=np.float64)
        except:
            return -99999
        
        if np.all(factor_values == 0) or np.all(np.isnan(factor_values)):
            return -99999
        
        ic_list = []
        dates = self.unique_dates
        
        for i in range(len(dates) - 1):
            date_t = dates[i]
            date_t1 = dates[i + 1]
            
            idx_t = self.date_to_indices[date_t]
            idx_t1 = self.date_to_indices[date_t1]
            
            f_t = factor_values[idx_t]
            r_t1 = self.future_ret[idx_t1]
            codes_t = self.codes[idx_t]
            codes_t1 = self.codes[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr, r_arr = np.array(f_list, dtype=np.float64), np.array(r_list, dtype=np.float64)
            
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, _ = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
            except:
                continue
        
        return np.mean(ic_list) if ic_list else -99999
    
    def calc_ic_series(self, factor_values):
        """返回每日IC序列（用于分析）"""
        try:
            factor_values = np.array(factor_values, dtype=np.float64)
        except:
            return []
        
        ic_list = []
        dates = self.unique_dates
        
        for i in range(len(dates) - 1):
            date_t = dates[i]
            date_t1 = dates[i + 1]
            
            idx_t = self.date_to_indices[date_t]
            idx_t1 = self.date_to_indices[date_t1]
            
            f_t = factor_values[idx_t]
            r_t1 = self.future_ret[idx_t1]
            codes_t = self.codes[idx_t]
            codes_t1 = self.codes[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr, r_arr = np.array(f_list, dtype=np.float64), np.array(r_list, dtype=np.float64)
            
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, _ = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
            except:
                continue
        
        return ic_list

# ==================== 5. DEAP设置 ====================

for attr in ['FitnessMax', 'Individual']:
    if hasattr(creator, attr):
        delattr(creator, attr)

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

# ==================== 6. 评估函数 ====================

def evaluate(individual, engine):
    try:
        factor = engine.evaluate_tree(individual)
        ic = engine.calc_ic(factor)
        return (ic,)
    except Exception as e:
        return (-99999,)

# ==================== 7. 因子相关性筛选 ====================

def filter_low_correlation(factors_dict, threshold=0.7):
    """
    筛选低相关因子
    factors_dict: {name: factor_values}
    threshold: 相关性阈值
    """
    if len(factors_dict) <= 1:
        return list(factors_dict.keys())
    
    names = list(factors_dict.keys())
    n = len(names)
    
    # 计算相关性矩阵
    corr_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i < j:
                f1 = factors_dict[names[i]]
                f2 = factors_dict[names[j]]
                # 计算相关系数
                mask = (~np.isnan(f1)) & (~np.isnan(f2)) & (~np.isinf(f1)) & (~np.isinf(f2))
                if mask.sum() > 10:
                    corr = np.corrcoef(f1[mask], f2[mask])[0, 1]
                    corr_matrix[i, j] = abs(corr)
                    corr_matrix[j, i] = abs(corr)
    
    # 贪心选择低相关因子
    selected = []
    remaining = list(range(n))
    
    # 按IC排序（假设第一个是最高IC）
    while remaining:
        # 选择剩余中相关性最低的
        if not selected:
            # 选择第一个
            selected.append(remaining.pop(0))
        else:
            # 计算与已选因子的平均相关性
            best_idx = remaining[0]
            best_score = float('inf')
            
            for idx in remaining:
                avg_corr = np.mean([corr_matrix[idx, sel] for sel in selected])
                if avg_corr < best_score:
                    best_score = avg_corr
                    best_idx = idx
            
            # 如果平均相关性低于阈值，选择
            if best_score < threshold:
                selected.append(best_idx)
            remaining.remove(best_idx)
    
    return [names[i] for i in selected]

# ==================== 8. 主程序（改进版） ====================

def run_gp_improved(df, pset_local, pop_size=100, n_gen=30, 
                    train_ratio=0.6, valid_ratio=0.2):
    """
    运行GP - 改进版
    - 三阶段分割（训练/验证/测试）
    - 早停机制
    - 因子相关性筛选
    """
    
    print("\n" + "="*60)
    print("🚀 LLM增强遗传编程 - 改进版")
    print("="*60)
    
    # 三阶段分割
    dates = df['trade_date'].unique()
    n_dates = len(dates)
    
    train_end = int(n_dates * train_ratio)
    valid_end = int(n_dates * (train_ratio + valid_ratio))
    
    train_dates = dates[:train_end]
    valid_dates = dates[train_end:valid_end]
    test_dates = dates[valid_end:]
    
    train_df = df[df['trade_date'].isin(train_dates)].copy()
    valid_df = df[df['trade_date'].isin(valid_dates)].copy()
    test_df = df[df['trade_date'].isin(test_dates)].copy()
    
    print(f"训练集: {len(train_dates)}天, {len(train_df)}行")
    print(f"验证集: {len(valid_dates)}天, {len(valid_df)}行")
    print(f"测试集: {len(test_dates)}天, {len(test_df)}行")
    
    # 构建训练引擎
    print("\n构建训练引擎...")
    train_engine = FastFactorEngine(train_df, name="Train")
    
    # 构建验证引擎（用于早停）
    print("\n构建验证引擎...")
    valid_engine = FastFactorEngine(valid_df, name="Valid")
    
    # 创建工具箱
    toolbox = base.Toolbox()
    toolbox.register("expr", gp.genHalfAndHalf, pset=pset_local, min_=1, max_=4)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", lambda ind: evaluate(ind, train_engine))
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("mate", gp.cxOnePoint)
    toolbox.register("mutate", gp.mutUniform, expr=partial(gp.genFull, pset=pset_local, min_=0, max_=2), pset=pset_local)
    
    # 限制树深度（防止过拟合）
    max_depth = 8
    toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=max_depth))
    toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=max_depth))
    
    # 测试评估函数
    print("\n测试评估函数...")
    try:
        test_expr = gp.Terminal('close', False, np.ndarray)
        test_ind = creator.Individual([test_expr])
        test_fit = evaluate(test_ind, train_engine)
        print(f"  close因子 IC={test_fit[0]:.4f}")
    except Exception as e:
        print(f"  ❌ 测试失败: {e}")
        return None
    
    # 初始化种群
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(10)
    
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("max", np.max)
    stats.register("min", np.min)
    
    print(f"\n种群: {pop_size}, 代数: {n_gen}")
    print(f"最大树深度: {max_depth}")
    print("-"*60)
    
    # 早停参数
    best_valid_ic = -99999
    patience = 5
    no_improve_count = 0
    best_individual = None
    
    # 记录进化历史
    history = {
        'train_ic': [],
        'valid_ic': [],
        'generation': []
    }
    
    # 进化
    for gen in range(n_gen):
        # 一代进化
        offspring = algorithms.varAnd(pop, toolbox, cxpb=0.6, mutpb=0.3)
        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit
        
        # 选择下一代
        pop = toolbox.select(offspring, len(pop))
        
        # 更新Hall of Fame
        for ind in pop:
            if ind not in hof:
                hof.update([ind])
        
        # 评估验证集
        if gen % 2 == 0 or gen == n_gen - 1:
            valid_ics = []
            for ind in hof[:5]:
                try:
                    factor = valid_engine.evaluate_tree(ind)
                    ic = valid_engine.calc_ic(factor)
                    if ic != -99999:
                        valid_ics.append(ic)
                except:
                    continue
            
            if valid_ics:
                avg_valid_ic = np.mean(valid_ics)
                history['valid_ic'].append(avg_valid_ic)
                history['train_ic'].append(np.mean([ind.fitness.values[0] for ind in hof[:5] if ind.fitness.values[0] != -99999]))
                history['generation'].append(gen)
                
                # 打印当前状态
                train_max = max([ind.fitness.values[0] for ind in pop if ind.fitness.values[0] != -99999], default=-99999)
                print(f"gen {gen:3d}: 训练max={train_max:.6f}, 验证IC={avg_valid_ic:.6f}")
                
                # 早停检查
                if avg_valid_ic > best_valid_ic:
                    best_valid_ic = avg_valid_ic
                    best_individual = hof[0]
                    no_improve_count = 0
                else:
                    no_improve_count += 1
                
                if no_improve_count >= patience:
                    print(f"\n⚠️ 早停: {patience}代无改善，停止进化")
                    break
            else:
                print(f"gen {gen:3d}: 验证IC计算失败")
    
    # 最终评估
    print("\n" + "="*60)
    print("📊 最终评估")
    print("="*60)
    
    # 构建测试引擎
    print("\n构建测试引擎...")
    test_engine = FastFactorEngine(test_df, name="Test")
    
    # 收集所有因子
    factors_dict = {}
    factor_names = []
    
    for i, ind in enumerate(hof[:10]):
        train_ic = ind.fitness.values[0]
        if train_ic == -99999:
            continue
        
        try:
            test_factor = test_engine.evaluate_tree(ind)
            test_ic = test_engine.calc_ic(test_factor)
            
            if test_ic != -99999:
                name = f"factor_{i+1}"
                factors_dict[name] = test_factor
                factor_names.append(name)
                print(f"{name}: 训练IC={train_ic:.6f}, 测试IC={test_ic:.6f}")
                print(f"  表达式: {str(ind)[:150]}")
                print()
        except Exception as e:
            print(f"因子{i+1}评估失败: {e}")
    
    # 相关性筛选
    if len(factors_dict) > 1:
        print("\n" + "="*60)
        print("📊 低相关因子筛选 (阈值=0.7)")
        print("="*60)
        
        selected = filter_low_correlation(factors_dict, threshold=0.7)
        print(f"筛选后保留 {len(selected)} 个低相关因子:")
        for name in selected:
            print(f"  ✅ {name}")
    else:
        selected = list(factors_dict.keys())
    
    # 保存结果
    if best_individual is not None:
        print("\n" + "="*60)
        print("💾 保存最优因子")
        print("="*60)
        
        # 创建保存目录
        save_dir = "factor_results"
        os.makedirs(save_dir, exist_ok=True)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = os.path.join(save_dir, f"best_factor_{timestamp}.pkl")
        
        result = {
            'individual': best_individual,
            'train_ic': best_individual.fitness.values[0],
            'valid_ic': best_valid_ic,
            'expression': str(best_individual),
            'history': history,
            'selected_factors': selected,
            'timestamp': timestamp
        }
        
        with open(save_path, 'wb') as f:
            pickle.dump(result, f)
        
        print(f"✅ 结果已保存到: {save_path}")
        
        # 保存文本报告
        report_path = os.path.join(save_dir, f"report_{timestamp}.txt")
        with open(report_path, 'w') as f:
            f.write("="*60 + "\n")
            f.write("因子挖掘报告\n")
            f.write("="*60 + "\n")
            f.write(f"时间: {timestamp}\n")
            f.write(f"训练集IC: {best_individual.fitness.values[0]:.6f}\n")
            f.write(f"验证集IC: {best_valid_ic:.6f}\n")
            f.write(f"表达式: {str(best_individual)}\n")
            f.write("\n低相关因子:\n")
            for name in selected:
                f.write(f"  {name}\n")
        
        print(f"✅ 报告已保存到: {report_path}")
    
    return best_individual, hof, history

# ==================== 9. 运行 ====================

if __name__ == "__main__":
    DATA_PATH = '../../extracted_data/stk_100.csv'
    try:
        df = load_data(DATA_PATH)
    except FileNotFoundError:
        DATA_PATH = '../extracted_data/stk_100.csv'
        df = load_data(DATA_PATH)
    
    # 运行改进版GP
    best, hof, history = run_gp_improved(
        df=df,
        pset_local=pset,
        pop_size=100,      # 增大种群
        n_gen=30,          # 增加代数
        train_ratio=0.6,   # 60%训练
        valid_ratio=0.2    # 20%验证
    )
    
    if best is not None:
        print("\n" + "="*60)
        print("✅ 因子挖掘完成！")
        print("="*60)
        print(f"最优因子训练IC: {best.fitness.values[0]:.6f}")
        print(f"最优因子表达式: {str(best)[:200]}")
    else:
        print("\n" + "="*60)
        print("❌ 因子挖掘失败")
        print("="*60)

✅ 加载 15 个金融基因
✅ GP原始集配置完成
📂 加载数据
📂 加载数据
原始数据: 24678行, 11列
检测到日期格式: 2024-01-02
✅ 使用格式: %Y-%m-%d
✅ 249天 × 100只股票 = 24578条记录

🚀 LLM增强遗传编程 - 改进版
训练集: 149天, 14689行
验证集: 50天, 4950行
测试集: 50天, 4939行

构建训练引擎...
计算金融基因 (Train)...
  [1/15] ✓ gap
  [2/15] ✓ vol_ratio
  [3/15] ✓ amplitude
  [4/15] ✓ upper_shadow
  [5/15] ✓ lower_shadow
  [6/15] ✓ momentum5
  [7/15] ✓ momentum10
  [8/15] ✓ momentum20
  [9/15] ✓ rsi
  [10/15] ✓ vol_price
  [11/15] ✓ macd
  [12/15] ✓ bbands_width
  [13/15] ✓ atr
  [14/15] ✓ vol_ma_ratio
  [15/15] ✓ price_to_ma
✅ Train 引擎就绪: 149天, 14689条记录

构建验证引擎...
计算金融基因 (Valid)...
  [1/15] ✓ gap
  [2/15] ✓ vol_ratio
  [3/15] ✓ amplitude
  [4/15] ✓ upper_shadow
  [5/15] ✓ lower_shadow
  [6/15] ✓ momentum5
  [7/15] ✓ momentum10
  [8/15] ✓ momentum20
  [9/15] ✓ rsi
  [10/15] ✓ vol_price
  [11/15] ✓ macd
  [12/15] ✓ bbands_width
  [13/15] ✓ atr
  [14/15] ✓ vol_ma_ratio
  [15/15] ✓ price_to_ma
✅ Valid 引擎就绪: 50天, 4950条记录

测试评估函数...
  close因子 IC=-0.0100

种群: 100, 代数: 30
最大树深度: 8
---------

In [ ]:
"""
LLM增强遗传编程因子挖掘系统 - 高性能优化版（修复IndexError）
"""

import numpy as np
import pandas as pd
from deap import base, creator, tools, gp, algorithms
import operator
import random
import warnings
from scipy.stats import spearmanr
from functools import partial
warnings.filterwarnings('ignore')

# ==================== 1. 数据加载 ====================

def parse_dates(df, date_col='trade_date'):
    sample = str(df[date_col].iloc[0])
    print(f"检测到日期格式: {sample}")
    
    formats = ['%Y-%m-%d', '%Y%m%d', '%Y/%m/%d', '%d-%m-%Y', '%d/%m/%Y', 'mixed']
    
    for fmt in formats:
        try:
            if fmt == 'mixed':
                df[date_col] = pd.to_datetime(df[date_col])
            else:
                df[date_col] = pd.to_datetime(df[date_col], format=fmt)
            print(f"✅ 使用格式: {fmt}")
            return df
        except:
            continue
    
    df[date_col] = pd.to_datetime(df[date_col])
    return df

def load_data(filepath):
    print("="*60)
    print("📂 加载数据")
    print("="*60)
    
    df = pd.read_csv(filepath)
    print(f"原始数据: {len(df)}行, {len(df.columns)}列")
    
    df = parse_dates(df, 'trade_date')
    df = df.sort_values(['trade_date', 'ts_code']).reset_index(drop=True)
    
    df['future_ret'] = df.groupby('ts_code')['close'].pct_change().shift(-1)
    df = df.dropna(subset=['future_ret', 'close', 'vol'])
    
    stocks = df['ts_code'].nunique()
    days = df['trade_date'].nunique()
    print(f"✅ {days}天 × {stocks}只股票 = {len(df)}条记录")
    
    return df

# ==================== 2. 金融基因 ====================

def to_float_array(x, n):
    if x is None:
        return np.zeros(n, dtype=np.float64)
    if isinstance(x, pd.Series):
        x = x.values
    if isinstance(x, np.ndarray):
        if x.dtype.type is np.str_ or x.dtype.type is np.object_:
            return np.zeros(n, dtype=np.float64)
        if np.issubdtype(x.dtype, np.datetime64):
            return np.zeros(n, dtype=np.float64)
        try:
            return x.astype(np.float64)
        except:
            return np.zeros(n, dtype=np.float64)
    if isinstance(x, (int, float)):
        return np.full(n, float(x), dtype=np.float64)
    try:
        return np.array(x, dtype=np.float64)
    except:
        return np.zeros(n, dtype=np.float64)

# --- 基因函数 ---
def gene_gap(df):
    result = df.groupby('ts_code')['open'].transform(lambda x: x / x.shift(1)) - 1
    return to_float_array(result, len(df))

def gene_vol_ratio(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(20, min_periods=10).mean()
    )
    return to_float_array(result, len(df))

def gene_amplitude(df):
    result = (df['high'] - df['low']) / df.groupby('ts_code')['close'].transform(lambda x: x.shift(1))
    return to_float_array(result, len(df))

def gene_upper_shadow(df):
    amp = df['high'] - df['low']
    result = (df['high'] - df['close']) / amp.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_lower_shadow(df):
    amp = df['high'] - df['low']
    result = (df['open'] - df['low']) / amp.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_momentum5(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(5) - 1)
    return to_float_array(result, len(df))

def gene_momentum10(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(10) - 1)
    return to_float_array(result, len(df))

def gene_momentum20(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(20) - 1)
    return to_float_array(result, len(df))

def gene_momentum60(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(60) - 1)
    return to_float_array(result, len(df))

def gene_rsi14(df):
    def _calc(group):
        diff = group.diff()
        gain = diff.clip(lower=0)
        loss = (-diff).clip(lower=0)
        avg_gain = gain.rolling(14, min_periods=7).mean()
        avg_loss = loss.rolling(14, min_periods=7).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_vol_price(df):
    price_pct = df.groupby('ts_code')['close'].transform(lambda x: x.pct_change())
    vol_pct = df.groupby('ts_code')['vol'].transform(lambda x: x.pct_change())
    result = price_pct / vol_pct.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_macd(df):
    def _calc(group):
        ema12 = group.ewm(span=12, adjust=False).mean()
        ema26 = group.ewm(span=26, adjust=False).mean()
        return ema12 - ema26
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_bbands_width(df):
    def _calc(group):
        ma = group.rolling(20, min_periods=10).mean()
        std = group.rolling(20, min_periods=10).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (upper - lower) / ma.replace(0, np.nan)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_bbands_position(df):
    def _calc(group):
        ma = group.rolling(20, min_periods=10).mean()
        std = group.rolling(20, min_periods=10).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (group - lower) / (upper - lower).replace(0, np.nan)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_atr(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x.pct_change().rolling(14, min_periods=7).std()
    )
    return to_float_array(result, len(df))

def gene_volume_ma_ratio(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(5, min_periods=3).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma5(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(5, min_periods=3).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma20(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(20, min_periods=10).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma60(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(60, min_periods=30).mean()
    )
    return to_float_array(result, len(df))

def gene_volume_breakout(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(20, min_periods=10).max()
    )
    return to_float_array(result, len(df))

def gene_price_breakout(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(20, min_periods=10).max()
    )
    return to_float_array(result, len(df))

# 基因字典
GENES = {
    'gap': gene_gap,
    'vol_ratio': gene_vol_ratio,
    'amplitude': gene_amplitude,
    'upper_shadow': gene_upper_shadow,
    'lower_shadow': gene_lower_shadow,
    'momentum5': gene_momentum5,
    'momentum10': gene_momentum10,
    'momentum20': gene_momentum20,
    'momentum60': gene_momentum60,
    'rsi': gene_rsi14,
    'vol_price': gene_vol_price,
    'macd': gene_macd,
    'bbands_width': gene_bbands_width,
    'bbands_position': gene_bbands_position,
    'atr': gene_atr,
    'vol_ma_ratio': gene_volume_ma_ratio,
    'price_to_ma5': gene_price_to_ma5,
    'price_to_ma20': gene_price_to_ma20,
    'price_to_ma60': gene_price_to_ma60,
    'vol_breakout': gene_volume_breakout,
    'price_breakout': gene_price_breakout,
}
print(f"✅ 加载 {len(GENES)} 个金融基因")

# ==================== 3. GP定义 ====================

def safe_div(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(np.abs(y) > 1e-10, x / y, 1.0)
        res = np.where(np.isinf(res), 1.0, res)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_add(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x + y

def safe_sub(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x - y

def safe_mul(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x * y

def safe_log(x):
    x = np.array(x, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(x > 1e-10, np.log(np.abs(x)), 0.0)
        res = np.where(np.isnan(res), 0.0, res)
        res = np.where(np.isinf(res), 0.0, res)
    return res

def safe_abs(x):
    x = np.array(x, dtype=np.float64)
    return np.abs(x)

def safe_sqrt(x):
    x = np.array(x, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(x >= 0, np.sqrt(x), 0.0)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_pow(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where((x >= 0) | (np.abs(y) < 1e-10), np.power(np.abs(x), y), 0.0)
        res = np.where(np.isinf(res), 0.0, res)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_max(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return np.maximum(x, y)

def safe_min(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return np.minimum(x, y)

def safe_sign(x):
    x = np.array(x, dtype=np.float64)
    return np.sign(x)

def rand_int():
    return random.choice([2, 3, 5, 7, 10, 15, 20, 30])

def rand_float():
    return random.uniform(0.3, 2.5)

# 创建原始集
pset = gp.PrimitiveSetTyped("MAIN", [], np.ndarray)

pset.addPrimitive(safe_div, [np.ndarray, np.ndarray], np.ndarray, name="div")
pset.addPrimitive(safe_add, [np.ndarray, np.ndarray], np.ndarray, name="add")
pset.addPrimitive(safe_sub, [np.ndarray, np.ndarray], np.ndarray, name="sub")
pset.addPrimitive(safe_mul, [np.ndarray, np.ndarray], np.ndarray, name="mul")
pset.addPrimitive(safe_log, [np.ndarray], np.ndarray, name="log")
pset.addPrimitive(safe_abs, [np.ndarray], np.ndarray, name="abs")
pset.addPrimitive(safe_sqrt, [np.ndarray], np.ndarray, name="sqrt")
pset.addPrimitive(safe_pow, [np.ndarray, np.ndarray], np.ndarray, name="pow")
pset.addPrimitive(safe_max, [np.ndarray, np.ndarray], np.ndarray, name="max")
pset.addPrimitive(safe_min, [np.ndarray, np.ndarray], np.ndarray, name="min")
pset.addPrimitive(safe_sign, [np.ndarray], np.ndarray, name="sign")

pset.addTerminal('close', np.ndarray)
pset.addTerminal('open', np.ndarray)
pset.addTerminal('high', np.ndarray)
pset.addTerminal('low', np.ndarray)
pset.addTerminal('volume', np.ndarray)

for name in GENES.keys():
    pset.addTerminal(name, np.ndarray)

pset.addEphemeralConstant("rand_int", rand_int, int)
pset.addEphemeralConstant("rand_float", rand_float, float)

print("✅ GP原始集配置完成")

# ==================== 4. 因子引擎 ====================

class FastFactorEngine:
    def __init__(self, df, name="Engine"):
        self.name = name
        self.df = df
        self.n = len(df)
        self.cache = {}
        
        self.cache.update({
            'close': df['close'].values.astype(np.float64),
            'open': df['open'].values.astype(np.float64),
            'high': df['high'].values.astype(np.float64),
            'low': df['low'].values.astype(np.float64),
            'volume': df['vol'].values.astype(np.float64),
        })
        
        print(f"计算金融基因 ({name})...")
        for i, (name_gene, func) in enumerate(GENES.items(), 1):
            try:
                result = func(df)
                self.cache[name_gene] = np.array(result, dtype=np.float64)
                print(f"  [{i}/{len(GENES)}] ✓ {name_gene}")
            except Exception as e:
                print(f"  [{i}/{len(GENES)}] ✗ {name_gene}: {e}")
                self.cache[name_gene] = np.zeros(self.n, dtype=np.float64)
        
        self.dates = df['trade_date'].values
        self.codes = df['ts_code'].values
        self.future_ret = df['future_ret'].values.astype(np.float64)
        
        self.date_to_indices = {}
        for idx, date in enumerate(self.dates):
            self.date_to_indices.setdefault(date, []).append(idx)
        
        self.unique_dates = sorted(self.date_to_indices.keys())
        print(f"✅ {name} 引擎就绪: {len(self.unique_dates)}天, {self.n}条记录")
    
    def evaluate_tree(self, expr_tree):
        try:
            if isinstance(expr_tree, gp.Primitive):
                return self._eval_primitive(expr_tree)
            elif isinstance(expr_tree, gp.Terminal):
                return self._eval_terminal(expr_tree)
            elif isinstance(expr_tree, list):
                return self._eval_list(expr_tree)
            else:
                return np.zeros(self.n, dtype=np.float64)
        except Exception:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_primitive(self, node):
        try:
            name = node.name
            args = [self.evaluate_tree(arg) for arg in node.args]
            
            args_clean = []
            for a in args:
                if isinstance(a, np.ndarray):
                    args_clean.append(a.astype(np.float64))
                else:
                    args_clean.append(np.array(a, dtype=np.float64))
            args = args_clean
            
            if name == 'add':
                return safe_add(args[0], args[1])
            elif name == 'sub':
                return safe_sub(args[0], args[1])
            elif name == 'mul':
                return safe_mul(args[0], args[1])
            elif name == 'div':
                return safe_div(args[0], args[1])
            elif name == 'log':
                return safe_log(args[0])
            elif name == 'abs':
                return safe_abs(args[0])
            elif name == 'sqrt':
                return safe_sqrt(args[0])
            elif name == 'pow':
                return safe_pow(args[0], args[1])
            elif name == 'max':
                return safe_max(args[0], args[1])
            elif name == 'min':
                return safe_min(args[0], args[1])
            elif name == 'sign':
                return safe_sign(args[0])
            else:
                return args[0] if args else np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_terminal(self, node):
        try:
            val = node.value
            if isinstance(val, str):
                if val in self.cache:
                    return self.cache[val].astype(np.float64)
                else:
                    return np.zeros(self.n, dtype=np.float64)
            elif isinstance(val, (int, float)):
                return np.full(self.n, float(val), dtype=np.float64)
            else:
                return np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_list(self, expr):
        try:
            if len(expr) >= 1 and callable(expr[0]):
                args = [self.evaluate_tree(arg) for arg in expr[1:]]
                args = [np.array(a, dtype=np.float64) if not isinstance(a, np.ndarray) else a for a in args]
                result = expr[0](*args)
                return np.array(result, dtype=np.float64)
            return self.evaluate_tree(expr[0]) if expr else np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def calc_ic(self, factor_values):
        try:
            factor_values = np.array(factor_values, dtype=np.float64)
        except:
            return -99999
        
        if np.all(factor_values == 0) or np.all(np.isnan(factor_values)):
            return -99999
        
        ic_list = []
        dates = self.unique_dates
        
        for i in range(len(dates) - 1):
            date_t = dates[i]
            date_t1 = dates[i + 1]
            
            idx_t = self.date_to_indices[date_t]
            idx_t1 = self.date_to_indices[date_t1]
            
            f_t = factor_values[idx_t]
            r_t1 = self.future_ret[idx_t1]
            codes_t = self.codes[idx_t]
            codes_t1 = self.codes[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr, r_arr = np.array(f_list, dtype=np.float64), np.array(r_list, dtype=np.float64)
            
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, _ = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
            except:
                continue
        
        return np.mean(ic_list) if ic_list else -99999
    
    def calc_ic_ir(self, factor_values):
        try:
            factor_values = np.array(factor_values, dtype=np.float64)
        except:
            return -99999, -99999
        
        if np.all(factor_values == 0) or np.all(np.isnan(factor_values)):
            return -99999, -99999
        
        ic_list = []
        dates = self.unique_dates
        
        for i in range(len(dates) - 1):
            date_t = dates[i]
            date_t1 = dates[i + 1]
            
            idx_t = self.date_to_indices[date_t]
            idx_t1 = self.date_to_indices[date_t1]
            
            f_t = factor_values[idx_t]
            r_t1 = self.future_ret[idx_t1]
            codes_t = self.codes[idx_t]
            codes_t1 = self.codes[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr, r_arr = np.array(f_list, dtype=np.float64), np.array(r_list, dtype=np.float64)
            
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, _ = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
            except:
                continue
        
        if len(ic_list) < 5:
            return -99999, -99999
        
        mean_ic = np.mean(ic_list)
        std_ic = np.std(ic_list)
        ir = mean_ic / std_ic if std_ic > 0 else -99999
        
        return mean_ic, ir

# ==================== 5. DEAP设置 ====================

for attr in ['FitnessMax', 'Individual']:
    if hasattr(creator, attr):
        delattr(creator, attr)

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

# ==================== 6. 评估函数 ====================

def evaluate_individual(individual, engine):
    try:
        factor = engine.evaluate_tree(individual)
        mean_ic, ir = engine.calc_ic_ir(factor)
        
        if mean_ic == -99999 or ir == -99999:
            return (-99999,)
        
        # 复合得分
        composite_score = mean_ic * (1 + max(0, ir) * 0.3)
        
        # 惩罚极端值
        factor_std = np.nanstd(factor)
        if factor_std > 10:
            composite_score *= 0.9
        if factor_std > 20:
            composite_score *= 0.8
        
        return (composite_score,)
    except Exception:
        return (-99999,)

# ==================== 7. 主程序 ====================

def run_gp_optimized(df, pset_local, pop_size=100, n_gen=30):
    print("\n" + "="*60)
    print("🚀 LLM增强遗传编程 - 优化版")
    print("="*60)
    
    # 三阶段分割
    dates = df['trade_date'].unique()
    n_dates = len(dates)
    
    train_end = int(n_dates * 0.6)
    valid_end = int(n_dates * 0.8)
    
    train_dates = dates[:train_end]
    valid_dates = dates[train_end:valid_end]
    test_dates = dates[valid_end:]
    
    train_df = df[df['trade_date'].isin(train_dates)].copy()
    valid_df = df[df['trade_date'].isin(valid_dates)].copy()
    test_df = df[df['trade_date'].isin(test_dates)].copy()
    
    print(f"训练集: {len(train_dates)}天, {len(train_df)}行")
    print(f"验证集: {len(valid_dates)}天, {len(valid_df)}行")
    print(f"测试集: {len(test_dates)}天, {len(test_df)}行")
    
    # 构建引擎
    print("\n构建训练引擎...")
    train_engine = FastFactorEngine(train_df, name="Train")
    
    print("\n构建验证引擎...")
    valid_engine = FastFactorEngine(valid_df, name="Valid")
    
    # 创建工具箱
    toolbox = base.Toolbox()
    toolbox.register("expr", gp.genHalfAndHalf, pset=pset_local, min_=1, max_=4)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", lambda ind: evaluate_individual(ind, train_engine))
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("mate", gp.cxOnePoint)
    toolbox.register("mutate", gp.mutUniform, expr=partial(gp.genFull, pset=pset_local, min_=0, max_=2), pset=pset_local)
    
    max_depth = 8
    toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=max_depth))
    toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=max_depth))
    
    # 测试评估
    print("\n测试评估函数...")
    try:
        test_expr = gp.Terminal('close', False, np.ndarray)
        test_ind = creator.Individual([test_expr])
        test_fit = evaluate_individual(test_ind, train_engine)
        print(f"  close因子 得分={test_fit[0]:.4f}")
    except Exception as e:
        print(f"  ❌ 测试失败: {e}")
        return None
    
    # 初始化
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(10)
    
    # 修复：安全统计函数，过滤无效个体
    def safe_stats(inds):
        valid_fits = [ind.fitness.values[0] for ind in inds if len(ind.fitness.values) > 0 and ind.fitness.values[0] != -99999]
        if valid_fits:
            return {
                'avg': np.mean(valid_fits),
                'max': np.max(valid_fits),
                'min': np.min(valid_fits),
                'std': np.std(valid_fits)
            }
        return {'avg': -99999, 'max': -99999, 'min': -99999, 'std': 0}
    
    print(f"\n种群: {pop_size}, 代数: {n_gen}")
    print(f"最大树深度: {max_depth}")
    print("-"*60)
    
    # 早停参数
    best_valid_ic = -99999
    patience = 6
    no_improve_count = 0
    best_individual = None
    
    # 进化
    for gen in range(n_gen):
        # 自适应变异率
        if gen < n_gen * 0.3:
            mutpb = 0.2
        elif gen < n_gen * 0.7:
            mutpb = 0.35
        else:
            mutpb = 0.15
        
        # 产生后代
        offspring = algorithms.varAnd(pop, toolbox, cxpb=0.6, mutpb=mutpb)
        
        # 评估
        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit
        
        # 精英保留
        pop.sort(key=lambda x: x.fitness.values[0] if len(x.fitness.values) > 0 else -99999, reverse=True)
        elite_count = max(1, int(pop_size * 0.15))
        elites = pop[:elite_count]
        
        pop = toolbox.select(offspring, pop_size - elite_count)
        pop.extend(elites)
        
        # 更新Hall of Fame
        for ind in pop:
            if ind not in hof and len(ind.fitness.values) > 0 and ind.fitness.values[0] != -99999:
                hof.update([ind])
        
        # 验证集评估
        if gen % 2 == 0 or gen == n_gen - 1:
            valid_results = []
            for ind in hof[:8]:
                try:
                    factor = valid_engine.evaluate_tree(ind)
                    mean_ic, ir = valid_engine.calc_ic_ir(factor)
                    if mean_ic != -99999:
                        valid_results.append((mean_ic, ir, ind))
                except:
                    continue
            
            # 打印统计信息
            stats_info = safe_stats(pop)
            print(f"gen {gen:3d}: 训练max={stats_info['max']:.6f}, avg={stats_info['avg']:.6f}")
            
            if valid_results:
                valid_results.sort(key=lambda x: x[0], reverse=True)
                avg_valid_ic = np.mean([r[0] for r in valid_results[:5]])
                best_valid_ic_curr = valid_results[0][0]
                print(f"         验证IC={avg_valid_ic:.6f}, 最优IC={best_valid_ic_curr:.6f}")
                
                if best_valid_ic_curr > best_valid_ic:
                    best_valid_ic = best_valid_ic_curr
                    best_individual = valid_results[0][2]
                    no_improve_count = 0
                    print(f"         ✅ 新最优!")
                else:
                    no_improve_count += 1
                
                if no_improve_count >= patience:
                    print(f"\n⚠️ 早停: {patience}代无改善")
                    break
    
    # 最终评估
    print("\n" + "="*60)
    print("📊 最终评估")
    print("="*60)
    
    print("\n构建测试引擎...")
    test_engine = FastFactorEngine(test_df, name="Test")
    
    print("\n最优因子测试结果:")
    if best_individual is not None:
        try:
            test_factor = test_engine.evaluate_tree(best_individual)
            test_ic, test_ir = test_engine.calc_ic_ir(test_factor)
            print(f"  验证集IC: {best_valid_ic:.6f}")
            print(f"  测试集IC: {test_ic:.6f}")
            print(f"  测试集IR: {test_ir:.3f}")
            print(f"  表达式: {str(best_individual)[:200]}")
        except Exception as e:
            print(f"  ❌ 评估失败: {e}")
    else:
        print("  ❌ 无最优因子，显示hof:")
        for i, ind in enumerate(hof[:3]):
            print(f"  因子{i+1}: {str(ind)[:100]}")
    
    # 显示所有因子
    print("\n所有候选因子 (Top 5):")
    for i, ind in enumerate(hof[:5]):
        if len(ind.fitness.values) == 0 or ind.fitness.values[0] == -99999:
            continue
        train_score = ind.fitness.values[0]
        try:
            test_factor = test_engine.evaluate_tree(ind)
            test_ic, test_ir = test_engine.calc_ic_ir(test_factor)
            if test_ic != -99999:
                print(f"\n因子{i+1}:")
                print(f"  训练得分: {train_score:.6f}")
                print(f"  测试IC: {test_ic:.6f}")
                print(f"  测试IR: {test_ir:.3f}")
                print(f"  表达式: {str(ind)[:150]}")
        except:
            continue
    
    return best_individual, hof

# ==================== 8. 运行 ====================

if __name__ == "__main__":
    DATA_PATH = '../../extracted_data/stk_100.csv'
    try:
        df = load_data(DATA_PATH)
    except FileNotFoundError:
        DATA_PATH = '../extracted_data/stk_100.csv'
        df = load_data(DATA_PATH)
    
    best, hof = run_gp_optimized(
        df=df,
        pset_local=pset,
        pop_size=100,
        n_gen=30
    )
    
    if best is not None:
        print("\n" + "="*60)
        print("✅ 因子挖掘完成！")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("❌ 因子挖掘失败")
        print("="*60)

✅ 加载 21 个金融基因
✅ GP原始集配置完成
📂 加载数据
📂 加载数据
原始数据: 24678行, 11列
检测到日期格式: 2024-01-02
✅ 使用格式: %Y-%m-%d
✅ 249天 × 100只股票 = 24578条记录

🚀 LLM增强遗传编程 - 优化版
训练集: 149天, 14689行
验证集: 50天, 4950行
测试集: 50天, 4939行

构建训练引擎...
计算金融基因 (Train)...
  [1/21] ✓ gap
  [2/21] ✓ vol_ratio
  [3/21] ✓ amplitude
  [4/21] ✓ upper_shadow
  [5/21] ✓ lower_shadow
  [6/21] ✓ momentum5
  [7/21] ✓ momentum10
  [8/21] ✓ momentum20
  [9/21] ✓ momentum60
  [10/21] ✓ rsi
  [11/21] ✓ vol_price
  [12/21] ✓ macd
  [13/21] ✓ bbands_width
  [14/21] ✓ bbands_position
  [15/21] ✓ atr
  [16/21] ✓ vol_ma_ratio
  [17/21] ✓ price_to_ma5
  [18/21] ✓ price_to_ma20
  [19/21] ✓ price_to_ma60
  [20/21] ✓ vol_breakout
  [21/21] ✓ price_breakout
✅ Train 引擎就绪: 149天, 14689条记录

构建验证引擎...
计算金融基因 (Valid)...
  [1/21] ✓ gap
  [2/21] ✓ vol_ratio
  [3/21] ✓ amplitude
  [4/21] ✓ upper_shadow
  [5/21] ✓ lower_shadow
  [6/21] ✓ momentum5
  [7/21] ✓ momentum10
  [8/21] ✓ momentum20
  [9/21] ✓ momentum60
  [10/21] ✓ rsi
  [11/21] ✓ vol_price
  [12/21] ✓ macd
  [13

In [1]:
"""
LLM增强遗传编程因子挖掘系统 - 高性能优化版（修复IndexError）
"""

import numpy as np
import pandas as pd
from deap import base, creator, tools, gp, algorithms
import operator
import random
import warnings
from scipy.stats import spearmanr
from functools import partial
warnings.filterwarnings('ignore')

# ==================== 1. 数据加载 ====================

def parse_dates(df, date_col='trade_date'):
    sample = str(df[date_col].iloc[0])
    print(f"检测到日期格式: {sample}")
    
    formats = ['%Y-%m-%d', '%Y%m%d', '%Y/%m/%d', '%d-%m-%Y', '%d/%m/%Y', 'mixed']
    
    for fmt in formats:
        try:
            if fmt == 'mixed':
                df[date_col] = pd.to_datetime(df[date_col])
            else:
                df[date_col] = pd.to_datetime(df[date_col], format=fmt)
            print(f"✅ 使用格式: {fmt}")
            return df
        except:
            continue
    
    df[date_col] = pd.to_datetime(df[date_col])
    return df

def load_data(filepath):
    print("="*60)
    print("📂 加载数据")
    print("="*60)
    
    df = pd.read_csv(filepath)
    print(f"原始数据: {len(df)}行, {len(df.columns)}列")
    
    df = parse_dates(df, 'trade_date')
    df = df.sort_values(['trade_date', 'ts_code']).reset_index(drop=True)
    
    df['future_ret'] = df.groupby('ts_code')['close'].pct_change().shift(-1)
    df = df.dropna(subset=['future_ret', 'close', 'vol'])
    
    stocks = df['ts_code'].nunique()
    days = df['trade_date'].nunique()
    print(f"✅ {days}天 × {stocks}只股票 = {len(df)}条记录")
    
    return df

# ==================== 2. 金融基因 ====================

def to_float_array(x, n):
    if x is None:
        return np.zeros(n, dtype=np.float64)
    if isinstance(x, pd.Series):
        x = x.values
    if isinstance(x, np.ndarray):
        if x.dtype.type is np.str_ or x.dtype.type is np.object_:
            return np.zeros(n, dtype=np.float64)
        if np.issubdtype(x.dtype, np.datetime64):
            return np.zeros(n, dtype=np.float64)
        try:
            return x.astype(np.float64)
        except:
            return np.zeros(n, dtype=np.float64)
    if isinstance(x, (int, float)):
        return np.full(n, float(x), dtype=np.float64)
    try:
        return np.array(x, dtype=np.float64)
    except:
        return np.zeros(n, dtype=np.float64)

# --- 基因函数 ---
def gene_gap(df):
    result = df.groupby('ts_code')['open'].transform(lambda x: x / x.shift(1)) - 1
    return to_float_array(result, len(df))

def gene_vol_ratio(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(20, min_periods=10).mean()
    )
    return to_float_array(result, len(df))

def gene_amplitude(df):
    result = (df['high'] - df['low']) / df.groupby('ts_code')['close'].transform(lambda x: x.shift(1))
    return to_float_array(result, len(df))

def gene_upper_shadow(df):
    amp = df['high'] - df['low']
    result = (df['high'] - df['close']) / amp.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_lower_shadow(df):
    amp = df['high'] - df['low']
    result = (df['open'] - df['low']) / amp.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_momentum5(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(5) - 1)
    return to_float_array(result, len(df))

def gene_momentum10(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(10) - 1)
    return to_float_array(result, len(df))

def gene_momentum20(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(20) - 1)
    return to_float_array(result, len(df))

def gene_momentum60(df):
    result = df.groupby('ts_code')['close'].transform(lambda x: x / x.shift(60) - 1)
    return to_float_array(result, len(df))

def gene_rsi14(df):
    def _calc(group):
        diff = group.diff()
        gain = diff.clip(lower=0)
        loss = (-diff).clip(lower=0)
        avg_gain = gain.rolling(14, min_periods=7).mean()
        avg_loss = loss.rolling(14, min_periods=7).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_vol_price(df):
    price_pct = df.groupby('ts_code')['close'].transform(lambda x: x.pct_change())
    vol_pct = df.groupby('ts_code')['vol'].transform(lambda x: x.pct_change())
    result = price_pct / vol_pct.replace(0, np.nan)
    return to_float_array(result, len(df))

def gene_macd(df):
    def _calc(group):
        ema12 = group.ewm(span=12, adjust=False).mean()
        ema26 = group.ewm(span=26, adjust=False).mean()
        return ema12 - ema26
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_bbands_width(df):
    def _calc(group):
        ma = group.rolling(20, min_periods=10).mean()
        std = group.rolling(20, min_periods=10).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (upper - lower) / ma.replace(0, np.nan)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_bbands_position(df):
    def _calc(group):
        ma = group.rolling(20, min_periods=10).mean()
        std = group.rolling(20, min_periods=10).std()
        upper = ma + 2 * std
        lower = ma - 2 * std
        return (group - lower) / (upper - lower).replace(0, np.nan)
    result = df.groupby('ts_code')['close'].transform(_calc)
    return to_float_array(result, len(df))

def gene_atr(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x.pct_change().rolling(14, min_periods=7).std()
    )
    return to_float_array(result, len(df))

def gene_volume_ma_ratio(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(5, min_periods=3).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma5(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(5, min_periods=3).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma20(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(20, min_periods=10).mean()
    )
    return to_float_array(result, len(df))

def gene_price_to_ma60(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(60, min_periods=30).mean()
    )
    return to_float_array(result, len(df))

def gene_volume_breakout(df):
    result = df.groupby('ts_code')['vol'].transform(
        lambda x: x / x.rolling(20, min_periods=10).max()
    )
    return to_float_array(result, len(df))

def gene_price_breakout(df):
    result = df.groupby('ts_code')['close'].transform(
        lambda x: x / x.rolling(20, min_periods=10).max()
    )
    return to_float_array(result, len(df))

# 基因字典
GENES = {
    'gap': gene_gap,
    'vol_ratio': gene_vol_ratio,
    'amplitude': gene_amplitude,
    'upper_shadow': gene_upper_shadow,
    'lower_shadow': gene_lower_shadow,
    'momentum5': gene_momentum5,
    'momentum10': gene_momentum10,
    'momentum20': gene_momentum20,
    'momentum60': gene_momentum60,
    'rsi': gene_rsi14,
    'vol_price': gene_vol_price,
    'macd': gene_macd,
    'bbands_width': gene_bbands_width,
    'bbands_position': gene_bbands_position,
    'atr': gene_atr,
    'vol_ma_ratio': gene_volume_ma_ratio,
    'price_to_ma5': gene_price_to_ma5,
    'price_to_ma20': gene_price_to_ma20,
    'price_to_ma60': gene_price_to_ma60,
    'vol_breakout': gene_volume_breakout,
    'price_breakout': gene_price_breakout,
}
print(f"✅ 加载 {len(GENES)} 个金融基因")

# ==================== 3. GP定义 ====================

def safe_div(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(np.abs(y) > 1e-10, x / y, 1.0)
        res = np.where(np.isinf(res), 1.0, res)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_add(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x + y

def safe_sub(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x - y

def safe_mul(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return x * y

def safe_log(x):
    x = np.array(x, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(x > 1e-10, np.log(np.abs(x)), 0.0)
        res = np.where(np.isnan(res), 0.0, res)
        res = np.where(np.isinf(res), 0.0, res)
    return res

def safe_abs(x):
    x = np.array(x, dtype=np.float64)
    return np.abs(x)

def safe_sqrt(x):
    x = np.array(x, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(x >= 0, np.sqrt(x), 0.0)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_pow(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where((x >= 0) | (np.abs(y) < 1e-10), np.power(np.abs(x), y), 0.0)
        res = np.where(np.isinf(res), 0.0, res)
        res = np.where(np.isnan(res), 0.0, res)
    return res

def safe_max(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return np.maximum(x, y)

def safe_min(x, y):
    x = np.array(x, dtype=np.float64)
    y = np.array(y, dtype=np.float64)
    if len(x) != len(y):
        min_len = min(len(x), len(y))
        x = x[:min_len]
        y = y[:min_len]
    return np.minimum(x, y)

def safe_sign(x):
    x = np.array(x, dtype=np.float64)
    return np.sign(x)

def rand_int():
    return random.choice([2, 3, 5, 7, 10, 15, 20, 30])

def rand_float():
    return random.uniform(0.3, 2.5)

# 创建原始集
pset = gp.PrimitiveSetTyped("MAIN", [], np.ndarray)

pset.addPrimitive(safe_div, [np.ndarray, np.ndarray], np.ndarray, name="div")
pset.addPrimitive(safe_add, [np.ndarray, np.ndarray], np.ndarray, name="add")
pset.addPrimitive(safe_sub, [np.ndarray, np.ndarray], np.ndarray, name="sub")
pset.addPrimitive(safe_mul, [np.ndarray, np.ndarray], np.ndarray, name="mul")
pset.addPrimitive(safe_log, [np.ndarray], np.ndarray, name="log")
pset.addPrimitive(safe_abs, [np.ndarray], np.ndarray, name="abs")
pset.addPrimitive(safe_sqrt, [np.ndarray], np.ndarray, name="sqrt")
pset.addPrimitive(safe_pow, [np.ndarray, np.ndarray], np.ndarray, name="pow")
pset.addPrimitive(safe_max, [np.ndarray, np.ndarray], np.ndarray, name="max")
pset.addPrimitive(safe_min, [np.ndarray, np.ndarray], np.ndarray, name="min")
pset.addPrimitive(safe_sign, [np.ndarray], np.ndarray, name="sign")

pset.addTerminal('close', np.ndarray)
pset.addTerminal('open', np.ndarray)
pset.addTerminal('high', np.ndarray)
pset.addTerminal('low', np.ndarray)
pset.addTerminal('volume', np.ndarray)

for name in GENES.keys():
    pset.addTerminal(name, np.ndarray)

pset.addEphemeralConstant("rand_int", rand_int, int)
pset.addEphemeralConstant("rand_float", rand_float, float)

print("✅ GP原始集配置完成")

# ==================== 4. 因子引擎 ====================

class FastFactorEngine:
    def __init__(self, df, name="Engine"):
        self.name = name
        self.df = df
        self.n = len(df)
        self.cache = {}
        
        self.cache.update({
            'close': df['close'].values.astype(np.float64),
            'open': df['open'].values.astype(np.float64),
            'high': df['high'].values.astype(np.float64),
            'low': df['low'].values.astype(np.float64),
            'volume': df['vol'].values.astype(np.float64),
        })
        
        print(f"计算金融基因 ({name})...")
        for i, (name_gene, func) in enumerate(GENES.items(), 1):
            try:
                result = func(df)
                self.cache[name_gene] = np.array(result, dtype=np.float64)
                print(f"  [{i}/{len(GENES)}] ✓ {name_gene}")
            except Exception as e:
                print(f"  [{i}/{len(GENES)}] ✗ {name_gene}: {e}")
                self.cache[name_gene] = np.zeros(self.n, dtype=np.float64)
        
        self.dates = df['trade_date'].values
        self.codes = df['ts_code'].values
        self.future_ret = df['future_ret'].values.astype(np.float64)
        
        self.date_to_indices = {}
        for idx, date in enumerate(self.dates):
            self.date_to_indices.setdefault(date, []).append(idx)
        
        self.unique_dates = sorted(self.date_to_indices.keys())
        print(f"✅ {name} 引擎就绪: {len(self.unique_dates)}天, {self.n}条记录")
    
    def evaluate_tree(self, expr_tree):
        try:
            if isinstance(expr_tree, gp.Primitive):
                return self._eval_primitive(expr_tree)
            elif isinstance(expr_tree, gp.Terminal):
                return self._eval_terminal(expr_tree)
            elif isinstance(expr_tree, list):
                return self._eval_list(expr_tree)
            else:
                return np.zeros(self.n, dtype=np.float64)
        except Exception:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_primitive(self, node):
        try:
            name = node.name
            args = [self.evaluate_tree(arg) for arg in node.args]
            
            args_clean = []
            for a in args:
                if isinstance(a, np.ndarray):
                    args_clean.append(a.astype(np.float64))
                else:
                    args_clean.append(np.array(a, dtype=np.float64))
            args = args_clean
            
            if name == 'add':
                return safe_add(args[0], args[1])
            elif name == 'sub':
                return safe_sub(args[0], args[1])
            elif name == 'mul':
                return safe_mul(args[0], args[1])
            elif name == 'div':
                return safe_div(args[0], args[1])
            elif name == 'log':
                return safe_log(args[0])
            elif name == 'abs':
                return safe_abs(args[0])
            elif name == 'sqrt':
                return safe_sqrt(args[0])
            elif name == 'pow':
                return safe_pow(args[0], args[1])
            elif name == 'max':
                return safe_max(args[0], args[1])
            elif name == 'min':
                return safe_min(args[0], args[1])
            elif name == 'sign':
                return safe_sign(args[0])
            else:
                return args[0] if args else np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_terminal(self, node):
        try:
            val = node.value
            if isinstance(val, str):
                if val in self.cache:
                    return self.cache[val].astype(np.float64)
                else:
                    return np.zeros(self.n, dtype=np.float64)
            elif isinstance(val, (int, float)):
                return np.full(self.n, float(val), dtype=np.float64)
            else:
                return np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def _eval_list(self, expr):
        try:
            if len(expr) >= 1 and callable(expr[0]):
                args = [self.evaluate_tree(arg) for arg in expr[1:]]
                args = [np.array(a, dtype=np.float64) if not isinstance(a, np.ndarray) else a for a in args]
                result = expr[0](*args)
                return np.array(result, dtype=np.float64)
            return self.evaluate_tree(expr[0]) if expr else np.zeros(self.n, dtype=np.float64)
        except:
            return np.zeros(self.n, dtype=np.float64)
    
    def calc_ic(self, factor_values):
        try:
            factor_values = np.array(factor_values, dtype=np.float64)
        except:
            return -99999
        
        if np.all(factor_values == 0) or np.all(np.isnan(factor_values)):
            return -99999
        
        ic_list = []
        dates = self.unique_dates
        
        for i in range(len(dates) - 1):
            date_t = dates[i]
            date_t1 = dates[i + 1]
            
            idx_t = self.date_to_indices[date_t]
            idx_t1 = self.date_to_indices[date_t1]
            
            f_t = factor_values[idx_t]
            r_t1 = self.future_ret[idx_t1]
            codes_t = self.codes[idx_t]
            codes_t1 = self.codes[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr, r_arr = np.array(f_list, dtype=np.float64), np.array(r_list, dtype=np.float64)
            
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, _ = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
            except:
                continue
        
        return np.mean(ic_list) if ic_list else -99999
    
    def calc_ic_ir(self, factor_values):
        try:
            factor_values = np.array(factor_values, dtype=np.float64)
        except:
            return -99999, -99999
        
        if np.all(factor_values == 0) or np.all(np.isnan(factor_values)):
            return -99999, -99999
        
        ic_list = []
        dates = self.unique_dates
        
        for i in range(len(dates) - 1):
            date_t = dates[i]
            date_t1 = dates[i + 1]
            
            idx_t = self.date_to_indices[date_t]
            idx_t1 = self.date_to_indices[date_t1]
            
            f_t = factor_values[idx_t]
            r_t1 = self.future_ret[idx_t1]
            codes_t = self.codes[idx_t]
            codes_t1 = self.codes[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr, r_arr = np.array(f_list, dtype=np.float64), np.array(r_list, dtype=np.float64)
            
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, _ = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
            except:
                continue
        
        if len(ic_list) < 5:
            return -99999, -99999
        
        mean_ic = np.mean(ic_list)
        std_ic = np.std(ic_list)
        ir = mean_ic / std_ic if std_ic > 0 else -99999
        
        return mean_ic, ir

# ==================== 5. DEAP设置 ====================

for attr in ['FitnessMax', 'Individual']:
    if hasattr(creator, attr):
        delattr(creator, attr)

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

# ==================== 6. 评估函数 ====================

def evaluate_individual(individual, engine):
    try:
        factor = engine.evaluate_tree(individual)
        mean_ic, ir = engine.calc_ic_ir(factor)
        
        if mean_ic == -99999 or ir == -99999:
            return (-99999,)
        
        # 复合得分
        composite_score = mean_ic * (1 + max(0, ir) * 0.3)
        
        # 惩罚极端值
        factor_std = np.nanstd(factor)
        if factor_std > 10:
            composite_score *= 0.9
        if factor_std > 20:
            composite_score *= 0.8
        
        return (composite_score,)
    except Exception:
        return (-99999,)

# ==================== 7. 主程序 ====================

def run_gp_optimized(df, pset_local, pop_size=100, n_gen=30):
    print("\n" + "="*60)
    print("🚀 LLM增强遗传编程 - 优化版")
    print("="*60)
    
    # 三阶段分割
    dates = df['trade_date'].unique()
    n_dates = len(dates)
    
    train_end = int(n_dates * 0.6)
    valid_end = int(n_dates * 0.8)
    
    train_dates = dates[:train_end]
    valid_dates = dates[train_end:valid_end]
    test_dates = dates[valid_end:]
    
    train_df = df[df['trade_date'].isin(train_dates)].copy()
    valid_df = df[df['trade_date'].isin(valid_dates)].copy()
    test_df = df[df['trade_date'].isin(test_dates)].copy()
    
    print(f"训练集: {len(train_dates)}天, {len(train_df)}行")
    print(f"验证集: {len(valid_dates)}天, {len(valid_df)}行")
    print(f"测试集: {len(test_dates)}天, {len(test_df)}行")
    
    # 构建引擎
    print("\n构建训练引擎...")
    train_engine = FastFactorEngine(train_df, name="Train")
    
    print("\n构建验证引擎...")
    valid_engine = FastFactorEngine(valid_df, name="Valid")
    
    # 创建工具箱
    toolbox = base.Toolbox()
    toolbox.register("expr", gp.genHalfAndHalf, pset=pset_local, min_=1, max_=4)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", lambda ind: evaluate_individual(ind, train_engine))
    toolbox.register("select", tools.selTournament, tournsize=3)
    toolbox.register("mate", gp.cxOnePoint)
    toolbox.register("mutate", gp.mutUniform, expr=partial(gp.genFull, pset=pset_local, min_=0, max_=2), pset=pset_local)
    
    max_depth = 8
    toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=max_depth))
    toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=max_depth))
    
    # 测试评估
    print("\n测试评估函数...")
    try:
        test_expr = gp.Terminal('close', False, np.ndarray)
        test_ind = creator.Individual([test_expr])
        test_fit = evaluate_individual(test_ind, train_engine)
        print(f"  close因子 得分={test_fit[0]:.4f}")
    except Exception as e:
        print(f"  ❌ 测试失败: {e}")
        return None
    
    # 初始化
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(10)
    
    # 修复：安全统计函数，过滤无效个体
    def safe_stats(inds):
        valid_fits = [ind.fitness.values[0] for ind in inds if len(ind.fitness.values) > 0 and ind.fitness.values[0] != -99999]
        if valid_fits:
            return {
                'avg': np.mean(valid_fits),
                'max': np.max(valid_fits),
                'min': np.min(valid_fits),
                'std': np.std(valid_fits)
            }
        return {'avg': -99999, 'max': -99999, 'min': -99999, 'std': 0}
    
    print(f"\n种群: {pop_size}, 代数: {n_gen}")
    print(f"最大树深度: {max_depth}")
    print("-"*60)
    
    # 早停参数
    best_valid_ic = -99999
    patience = 6
    no_improve_count = 0
    best_individual = None
    
    # 进化
    for gen in range(n_gen):
        # 自适应变异率
        if gen < n_gen * 0.3:
            mutpb = 0.2
        elif gen < n_gen * 0.7:
            mutpb = 0.35
        else:
            mutpb = 0.15
        
        # 产生后代
        offspring = algorithms.varAnd(pop, toolbox, cxpb=0.6, mutpb=mutpb)
        
        # 评估
        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit
        
        # 精英保留
        pop.sort(key=lambda x: x.fitness.values[0] if len(x.fitness.values) > 0 else -99999, reverse=True)
        elite_count = max(1, int(pop_size * 0.15))
        elites = pop[:elite_count]
        
        pop = toolbox.select(offspring, pop_size - elite_count)
        pop.extend(elites)
        
        # 更新Hall of Fame
        for ind in pop:
            if ind not in hof and len(ind.fitness.values) > 0 and ind.fitness.values[0] != -99999:
                hof.update([ind])
        
        # 验证集评估
        if gen % 2 == 0 or gen == n_gen - 1:
            valid_results = []
            for ind in hof[:8]:
                try:
                    factor = valid_engine.evaluate_tree(ind)
                    mean_ic, ir = valid_engine.calc_ic_ir(factor)
                    if mean_ic != -99999:
                        valid_results.append((mean_ic, ir, ind))
                except:
                    continue
            
            # 打印统计信息
            stats_info = safe_stats(pop)
            print(f"gen {gen:3d}: 训练max={stats_info['max']:.6f}, avg={stats_info['avg']:.6f}")
            
            if valid_results:
                valid_results.sort(key=lambda x: x[0], reverse=True)
                avg_valid_ic = np.mean([r[0] for r in valid_results[:5]])
                best_valid_ic_curr = valid_results[0][0]
                print(f"         验证IC={avg_valid_ic:.6f}, 最优IC={best_valid_ic_curr:.6f}")
                
                if best_valid_ic_curr > best_valid_ic:
                    best_valid_ic = best_valid_ic_curr
                    best_individual = valid_results[0][2]
                    no_improve_count = 0
                    print(f"         ✅ 新最优!")
                else:
                    no_improve_count += 1
                
                if no_improve_count >= patience:
                    print(f"\n⚠️ 早停: {patience}代无改善")
                    break
    
    # 最终评估
    print("\n" + "="*60)
    print("📊 最终评估")
    print("="*60)
    
    print("\n构建测试引擎...")
    test_engine = FastFactorEngine(test_df, name="Test")
    
    print("\n最优因子测试结果:")
    if best_individual is not None:
        try:
            test_factor = test_engine.evaluate_tree(best_individual)
            test_ic, test_ir = test_engine.calc_ic_ir(test_factor)
            print(f"  验证集IC: {best_valid_ic:.6f}")
            print(f"  测试集IC: {test_ic:.6f}")
            print(f"  测试集IR: {test_ir:.3f}")
            print(f"  表达式: {str(best_individual)[:200]}")
        except Exception as e:
            print(f"  ❌ 评估失败: {e}")
    else:
        print("  ❌ 无最优因子，显示hof:")
        for i, ind in enumerate(hof[:3]):
            print(f"  因子{i+1}: {str(ind)[:100]}")
    
    # 显示所有因子
    print("\n所有候选因子 (Top 5):")
    for i, ind in enumerate(hof[:5]):
        if len(ind.fitness.values) == 0 or ind.fitness.values[0] == -99999:
            continue
        train_score = ind.fitness.values[0]
        try:
            test_factor = test_engine.evaluate_tree(ind)
            test_ic, test_ir = test_engine.calc_ic_ir(test_factor)
            if test_ic != -99999:
                print(f"\n因子{i+1}:")
                print(f"  训练得分: {train_score:.6f}")
                print(f"  测试IC: {test_ic:.6f}")
                print(f"  测试IR: {test_ir:.3f}")
                print(f"  表达式: {str(ind)[:150]}")
        except:
            continue
    
    return best_individual, hof

# ==================== 8. 运行 ====================

if __name__ == "__main__":
    DATA_PATH = '../../extracted_data/stk_100.csv'
    try:
        df = load_data(DATA_PATH)
    except FileNotFoundError:
        DATA_PATH = '../extracted_data/stk_100.csv'
        df = load_data(DATA_PATH)
    
    best, hof = run_gp_optimized(
        df=df,
        pset_local=pset,
        pop_size=100,
        n_gen=30
    )
    
    if best is not None:
        print("\n" + "="*60)
        print("✅ 因子挖掘完成！")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("❌ 因子挖掘失败")
        print("="*60)

✅ 加载 21 个金融基因
✅ GP原始集配置完成
📂 加载数据
📂 加载数据
原始数据: 24678行, 11列
检测到日期格式: 2024-01-02
✅ 使用格式: %Y-%m-%d
✅ 249天 × 100只股票 = 24578条记录

🚀 LLM增强遗传编程 - 优化版
训练集: 149天, 14689行
验证集: 50天, 4950行
测试集: 50天, 4939行

构建训练引擎...
计算金融基因 (Train)...
  [1/21] ✓ gap
  [2/21] ✓ vol_ratio
  [3/21] ✓ amplitude
  [4/21] ✓ upper_shadow
  [5/21] ✓ lower_shadow
  [6/21] ✓ momentum5
  [7/21] ✓ momentum10
  [8/21] ✓ momentum20
  [9/21] ✓ momentum60
  [10/21] ✓ rsi
  [11/21] ✓ vol_price
  [12/21] ✓ macd
  [13/21] ✓ bbands_width
  [14/21] ✓ bbands_position
  [15/21] ✓ atr
  [16/21] ✓ vol_ma_ratio
  [17/21] ✓ price_to_ma5
  [18/21] ✓ price_to_ma20
  [19/21] ✓ price_to_ma60
  [20/21] ✓ vol_breakout
  [21/21] ✓ price_breakout
✅ Train 引擎就绪: 149天, 14689条记录

构建验证引擎...
计算金融基因 (Valid)...
  [1/21] ✓ gap
  [2/21] ✓ vol_ratio
  [3/21] ✓ amplitude
  [4/21] ✓ upper_shadow
  [5/21] ✓ lower_shadow
  [6/21] ✓ momentum5
  [7/21] ✓ momentum10
  [8/21] ✓ momentum20
  [9/21] ✓ momentum60
  [10/21] ✓ rsi
  [11/21] ✓ vol_price
  [12/21] ✓ macd
  [13

In [ ]:
"""
因子验证模块
功能：
1. 统计显著性检验（RankIC + p值）
2. IR计算
3. 时间序列稳定性分析
4. 分层回测（分组收益单调性）
"""

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, ttest_1samp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. 因子验证类 ====================

class FactorValidator:
    """
    因子验证器
    对挖掘出的因子进行全方位的有效性验证
    """
    
    def __init__(self, df, factor_values, factor_name="Factor"):
        """
        参数:
        df: DataFrame, 包含 'trade_date', 'ts_code', 'future_ret' 等列
        factor_values: numpy array, 因子值序列（与df长度一致）
        factor_name: str, 因子名称
        """
        self.df = df.copy()
        self.factor_values = np.array(factor_values)
        self.factor_name = factor_name
        self.n = len(df)
        
        # 按日期分组
        self.df['factor'] = self.factor_values
        self.dates = self.df['trade_date'].unique()
        self.dates = np.sort(self.dates)
        
        # 构建日期到索引的映射
        self.date_to_idx = {}
        for idx, date in enumerate(self.df['trade_date']):
            self.date_to_idx.setdefault(date, []).append(idx)
        
        # 存储计算结果
        self.results = {}
    
    # ==================== 1.1 RankIC计算 ====================
    
    def calculate_ic_series(self):
        """计算每日IC序列"""
        ic_list = []
        date_list = []
        
        for i in range(len(self.dates) - 1):
            date_t = self.dates[i]
            date_t1 = self.dates[i + 1]
            
            idx_t = self.date_to_idx[date_t]
            idx_t1 = self.date_to_idx[date_t1]
            
            f_t = self.factor_values[idx_t]
            r_t1 = self.df['future_ret'].values[idx_t1]
            codes_t = self.df['ts_code'].values[idx_t]
            codes_t1 = self.df['ts_code'].values[idx_t1]
            
            # 按股票代码对齐
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            f_list, r_list = [], []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        f_list.append(f_t[j])
                        r_list.append(r_val)
            
            if len(f_list) < 10:
                continue
            
            f_arr = np.array(f_list)
            r_arr = np.array(r_list)
            
            # 去极值
            q_low, q_high = np.percentile(f_arr, [1, 99])
            mask = (f_arr >= q_low) & (f_arr <= q_high)
            
            if mask.sum() < 10:
                continue
            
            try:
                ic, p_val = spearmanr(f_arr[mask], r_arr[mask])
                if not np.isnan(ic):
                    ic_list.append(ic)
                    date_list.append(date_t)
            except:
                continue
        
        self.results['ic_series'] = ic_list
        self.results['ic_dates'] = date_list
        return ic_list, date_list
    
    # ==================== 1.2 统计显著性 ====================
    
    def test_statistical_significance(self):
        """
        统计显著性检验
        检验IC均值是否显著大于0
        """
        ic_series = self.results.get('ic_series')
        if ic_series is None:
            ic_series, _ = self.calculate_ic_series()
        
        if len(ic_series) < 5:
            return {
                'mean_ic': np.nan,
                'std_ic': np.nan,
                'ir': np.nan,
                't_stat': np.nan,
                'p_value': np.nan,
                'significant': False,
                'is_valid': False
            }
        
        mean_ic = np.mean(ic_series)
        std_ic = np.std(ic_series)
        ir = mean_ic / std_ic if std_ic > 0 else np.nan
        
        # t检验：检验IC均值是否显著大于0
        t_stat, p_value = ttest_1samp(ic_series, 0)
        
        # 判断标准：IC > 0.03 且 p值 < 0.05
        significant = (mean_ic > 0.03) and (p_value < 0.05)
        
        self.results['statistical'] = {
            'mean_ic': mean_ic,
            'std_ic': std_ic,
            'ir': ir,
            't_stat': t_stat,
            'p_value': p_value,
            'significant': significant,
            'is_valid': significant and (ir > 0.5 if not np.isnan(ir) else False)
        }
        
        return self.results['statistical']
    
    # ==================== 1.3 时间序列稳定性 ====================
    
    def test_temporal_stability(self):
        """
        时间序列稳定性检验
        1. IC序列是否有明显衰减
        2. 分年份IC是否均为正
        """
        ic_series = self.results.get('ic_series')
        ic_dates = self.results.get('ic_dates')
        
        if ic_series is None or len(ic_series) < 10:
            return {'is_stable': False, 'yearly_ics': {}, 'trend': np.nan}
        
        # 1. 趋势检验：将IC序列分为前后两半
        n = len(ic_series)
        half = n // 2
        first_half_mean = np.mean(ic_series[:half])
        second_half_mean = np.mean(ic_series[half:])
        decay = first_half_mean - second_half_mean
        
        # 2. 分年份IC
        yearly_ics = {}
        for ic, date in zip(ic_series, ic_dates):
            year = pd.to_datetime(date).year
            yearly_ics.setdefault(year, []).append(ic)
        
        yearly_means = {year: np.mean(ics) for year, ics in yearly_ics.items()}
        
        # 判断：后半年IC不能显著低于前半年，且每年IC均为正
        all_years_positive = all(v > 0 for v in yearly_means.values())
        no_significant_decay = decay < 0.01  # 衰减小于1%
        
        is_stable = all_years_positive and no_significant_decay
        
        self.results['temporal'] = {
            'is_stable': is_stable,
            'yearly_ics': yearly_means,
            'first_half_ic': first_half_mean,
            'second_half_ic': second_half_mean,
            'decay': decay,
            'all_years_positive': all_years_positive
        }
        
        return self.results['temporal']
    
    # ==================== 1.4 分层回测 ====================
    
    def stratified_backtest(self, n_groups=10):
        """
        分层回测
        按因子值分为N组，计算每组未来收益
        """
        # 按日期分组计算
        group_returns = {i: [] for i in range(n_groups)}
        
        for date in self.dates:
            idx = self.date_to_idx[date]
            if len(idx) < n_groups:
                continue
            
            f_t = self.factor_values[idx]
            codes_t = self.df['ts_code'].values[idx]
            
            # 获取下一日收益
            date_idx = list(self.dates).index(date)
            if date_idx >= len(self.dates) - 1:
                continue
            
            date_t1 = self.dates[date_idx + 1]
            idx_t1 = self.date_to_idx[date_t1]
            r_t1 = self.df['future_ret'].values[idx_t1]
            codes_t1 = self.df['ts_code'].values[idx_t1]
            
            ret_dict = {codes_t1[j]: r_t1[j] for j in range(len(codes_t1))}
            
            # 构建（因子值，收益）对
            pairs = []
            for j, code in enumerate(codes_t):
                if code in ret_dict:
                    r_val = ret_dict[code]
                    if not np.isnan(f_t[j]) and not np.isnan(r_val):
                        pairs.append((f_t[j], r_val))
            
            if len(pairs) < n_groups:
                continue
            
            pairs.sort(key=lambda x: x[0])
            
            # 等量分组
            group_size = len(pairs) // n_groups
            for g in range(n_groups):
                start = g * group_size
                end = (g + 1) * group_size if g < n_groups - 1 else len(pairs)
                group_returns[g].extend([p[1] for p in pairs[start:end]])
        
        # 计算每组平均收益
        group_means = [np.mean(group_returns[i]) if group_returns[i] else np.nan for i in range(n_groups)]
        
        # 计算单调性：Top组收益 > Bottom组收益
        top_mean = group_means[0] if not np.isnan(group_means[0]) else -999
        bottom_mean = group_means[-1] if not np.isnan(group_means[-1]) else -999
        is_monotonic = top_mean > bottom_mean
        
        # 计算各组的t值（相对于0）
        group_t_stats = []
        for i in range(n_groups):
            if len(group_returns[i]) > 5:
                t_stat, p_val = ttest_1samp(group_returns[i], 0)
                group_t_stats.append((t_stat, p_val))
            else:
                group_t_stats.append((np.nan, np.nan))
        
        self.results['backtest'] = {
            'n_groups': n_groups,
            'group_means': group_means,
            'group_returns': group_returns,
            'top_mean': top_mean,
            'bottom_mean': bottom_mean,
            'is_monotonic': is_monotonic,
            'group_t_stats': group_t_stats,
            'spread': top_mean - bottom_mean  # Top-Bottom收益差
        }
        
        return self.results['backtest']
    
    # ==================== 1.5 完整验证报告 ====================
    
    def generate_full_report(self, n_groups=10):
        """
        生成完整的因子验证报告
        """
        print("="*70)
        print(f"📊 因子验证报告: {self.factor_name}")
        print("="*70)
        
        # 1. IC序列
        ic_series, _ = self.calculate_ic_series()
        print(f"\n【1. IC序列统计】")
        print(f"   IC序列长度: {len(ic_series)}")
        print(f"   IC均值: {np.mean(ic_series):.6f}")
        print(f"   IC标准差: {np.std(ic_series):.6f}")
        
        # 2. 统计显著性
        stat = self.test_statistical_significance()
        print(f"\n【2. 统计显著性检验】")
        print(f"   均值IC: {stat['mean_ic']:.6f}")
        print(f"   IR: {stat['ir']:.3f}")
        print(f"   t统计量: {stat['t_stat']:.3f}")
        print(f"   p值: {stat['p_value']:.6f}")
        print(f"   是否显著 (IC>0.03, p<0.05): {'✅ 是' if stat['significant'] else '❌ 否'}")
        print(f"   是否有效 (IC>0.03 & IR>0.5): {'✅ 是' if stat['is_valid'] else '❌ 否'}")
        
        # 3. 时间序列稳定性
        temporal = self.test_temporal_stability()
        print(f"\n【3. 时间序列稳定性】")
        print(f"   前半段IC均值: {temporal['first_half_ic']:.6f}")
        print(f"   后半段IC均值: {temporal['second_half_ic']:.6f}")
        print(f"   衰减值: {temporal['decay']:.6f}")
        print(f"   各年份IC: {temporal['yearly_ics']}")
        print(f"   各年份IC均为正: {'✅ 是' if temporal['all_years_positive'] else '❌ 否'}")
        print(f"   整体稳定: {'✅ 是' if temporal['is_stable'] else '❌ 否'}")
        
        # 4. 分层回测
        bt = self.stratified_backtest(n_groups)
        print(f"\n【4. 分层回测 (分组数={n_groups})】")
        print(f"   Top组收益: {bt['top_mean']:.6f}")
        print(f"   Bottom组收益: {bt['bottom_mean']:.6f}")
        print(f"   Top-Bottom收益差: {bt['spread']:.6f}")
        print(f"   分组收益单调性: {'✅ 是' if bt['is_monotonic'] else '❌ 否'}")
        
        # 打印分组收益
        print(f"\n   各分组收益:")
        for i, mean_ret in enumerate(bt['group_means']):
            if not np.isnan(mean_ret):
                print(f"     组{i+1:2d}: {mean_ret:.6f}")
        
        # 5. 综合评级
        print(f"\n【5. 综合评级】")
        criteria = {
            '统计显著性 (IC>0.03)': stat['significant'],
            'IR>0.5': stat['ir'] > 0.5 if not np.isnan(stat['ir']) else False,
            'IC无衰减': temporal['decay'] < 0.01,
            '各年IC为正': temporal['all_years_positive'],
            '分组单调性': bt['is_monotonic'],
            'Top-Bottom为正': bt['spread'] > 0
        }
        
        passed = sum(criteria.values())
        total = len(criteria)
        print(f"   通过标准: {passed}/{total}")
        
        if passed >= 4:
            print(f"   ✅ 综合评级: 优秀 - 因子具有显著预测能力")
        elif passed >= 3:
            print(f"   ✅ 综合评级: 良好 - 因子有一定预测能力")
        elif passed >= 2:
            print(f"   ⚠️ 综合评级: 一般 - 因子预测能力有限")
        else:
            print(f"   ❌ 综合评级: 较差 - 因子无效或过拟合")
        
        self.results['summary'] = {
            'criteria': criteria,
            'passed': passed,
            'total': total,
            'rating': '优秀' if passed >= 4 else '良好' if passed >= 3 else '一般' if passed >= 2 else '较差'
        }
        
        return self.results
    
    # ==================== 1.6 可视化 ====================
    
    def plot_results(self, n_groups=10, save_path=None):
        """
        绘制验证结果图
        """
        if not self.results:
            self.generate_full_report(n_groups)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'因子验证图表: {self.factor_name}', fontsize=14, fontweight='bold')
        
        # 图1: IC时间序列
        ax1 = axes[0, 0]
        ic_series = self.results.get('ic_series', [])
        ic_dates = self.results.get('ic_dates', [])
        if ic_series and ic_dates:
            ax1.plot(ic_dates, ic_series, 'b-', alpha=0.7, linewidth=0.8)
            ax1.axhline(y=0, color='red', linestyle='--', linewidth=1)
            ax1.axhline(y=np.mean(ic_series), color='green', linestyle='--', linewidth=1, label=f'均值={np.mean(ic_series):.4f}')
            ax1.set_title('IC时间序列')
            ax1.set_xlabel('日期')
            ax1.set_ylabel('IC')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
        
        # 图2: IC直方图
        ax2 = axes[0, 1]
        if ic_series:
            ax2.hist(ic_series, bins=30, edgecolor='black', alpha=0.7, color='blue')
            ax2.axvline(x=0, color='red', linestyle='--', linewidth=1)
            ax2.axvline(x=np.mean(ic_series), color='green', linestyle='--', linewidth=1, label=f'均值={np.mean(ic_series):.4f}')
            ax2.set_title('IC分布直方图')
            ax2.set_xlabel('IC')
            ax2.set_ylabel('频数')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
        
        # 图3: 分层回测
        ax3 = axes[1, 0]
        bt = self.results.get('backtest', {})
        group_means = bt.get('group_means', [])
        if group_means:
            x = range(1, len(group_means) + 1)
            bars = ax3.bar(x, group_means, color='steelblue', edgecolor='black')
            ax3.axhline(y=0, color='red', linestyle='--', linewidth=1)
            ax3.set_title(f'分层回测 (分组数={len(group_means)})')
            ax3.set_xlabel('分组 (1=最高因子值)')
            ax3.set_ylabel('平均收益率')
            ax3.grid(True, alpha=0.3, axis='y')
            
            # 添加数值标签
            for bar, val in zip(bars, group_means):
                if not np.isnan(val):
                    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001, 
                            f'{val:.4f}', ha='center', va='bottom', fontsize=8)
        
        # 图4: 各年IC
        ax4 = axes[1, 1]
        temporal = self.results.get('temporal', {})
        yearly_ics = temporal.get('yearly_ics', {})
        if yearly_ics:
            years = list(yearly_ics.keys())
            ics = list(yearly_ics.values())
            bars = ax4.bar(years, ics, color='coral', edgecolor='black')
            ax4.axhline(y=0, color='red', linestyle='--', linewidth=1)
            ax4.set_title('各年IC均值')
            ax4.set_xlabel('年份')
            ax4.set_ylabel('IC均值')
            ax4.grid(True, alpha=0.3, axis='y')
            
            for bar, val in zip(bars, ics):
                if not np.isnan(val):
                    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, 
                            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        
        # if save_path:
        #     plt.savefig(save_path, dpi=150, bbox_inches='tight')
        #     print(f"✅ 图表已保存: {save_path}")
        
        plt.show()
        
        return fig


# ==================== 2. 使用示例 ====================

def validate_factor(df, factor_values, factor_name="Factor", n_groups=10, plot=True):
    """
    便捷函数：一键验证因子
    """
    validator = FactorValidator(df, factor_values, factor_name)
    report = validator.generate_full_report(n_groups)
    
    if plot:
        validator.plot_results(n_groups)
    
    return validator, report

# ==================== 3. 与挖掘系统集成 ====================

def validate_best_factor(df, hof, pset, engine, n_groups=10):
    """
    验证Hall of Fame中的最优因子
    """
    if hof is None or len(hof) == 0:
        print("❌ 没有可验证的因子")
        return None
    
    results = {}
    
    for i, ind in enumerate(hof[:5]):
        try:
            # 计算因子值
            factor_values = engine.evaluate_tree(ind)
            
            # 验证
            name = f"Factor_{i+1}"
            validator = FactorValidator(df, factor_values, name)
            report = validator.generate_full_report(n_groups)
            results[name] = {
                'validator': validator,
                'report': report,
                'expression': str(ind)
            }
            
            print(f"\n{'='*70}")
            print(f"表达式: {str(ind)[:100]}...")
            
        except Exception as e:
            print(f"❌ 因子{i+1}验证失败: {e}")
            continue
    
    return results


# ==================== 4. 主程序 ====================

if __name__ == "__main__":
    print("="*70)
    print("📊 因子验证模块")
    print("="*70)
    print("\n使用说明:")
    print("1. 从主程序加载数据后，获取因子值")
    print("2. 调用 validate_factor() 进行验证")
    print("3. 或调用 validate_best_factor() 验证最优因子")
    print("\n示例:")
    print("""
# 从主程序获取测试集

test_engine = FastFactorEngine(test_df, name="Test")
best_ind = hof[0]  # 最优个体

# 计算因子值
factor_values = test_engine.evaluate_tree(best_ind)

# 验证
validator = validate_factor(test_df, factor_values, "Best_Factor", n_groups=10)
    """)
    


📊 因子验证模块

使用说明:
1. 从主程序加载数据后，获取因子值
2. 调用 validate_factor() 进行验证
3. 或调用 validate_best_factor() 验证最优因子

示例:

# 从主程序获取测试集

test_engine = FastFactorEngine(test_df, name="Test")
best_ind = hof[0]  # 最优个体

# 计算因子值
factor_values = test_engine.evaluate_tree(best_ind)

# 验证
validator = validate_factor(test_df, factor_values, "Best_Factor", n_groups=10)
    


In [3]:
if __name__ == "__main__":
    DATA_PATH = '../../extracted_data/stk_100.csv'
    try:
        df = load_data(DATA_PATH)
    except FileNotFoundError:
        DATA_PATH = '../extracted_data/stk_100.csv'
        df = load_data(DATA_PATH)
    
    best, hof = run_gp_optimized(
        df=df,
        pset_local=pset,
        pop_size=100,
        n_gen=30
    )
    
    if best is not None:
        print("\n" + "="*60)
        print("✅ 因子挖掘完成！")
        print("="*60)
        
        # ========== 添加验证代码 ==========
        print("\n" + "="*60)
        print("🔍 开始因子验证")
        print("="*60)
        
        # 重新获取测试集（需要从run_gp_optimized返回test_df）
        # 或者直接使用全局df重新分割
        dates = df['trade_date'].unique()
        n_dates = len(dates)
        train_end = int(n_dates * 0.6)
        valid_end = int(n_dates * 0.8)
        test_dates = dates[valid_end:]
        test_df = df[df['trade_date'].isin(test_dates)].copy()
        
        # 构建测试引擎
        test_engine = FastFactorEngine(test_df, name="Test")
        
        # 验证Top 5因子
        for i, ind in enumerate(hof[:5]):
            if len(ind.fitness.values) == 0 or ind.fitness.values[0] == -99999:
                continue
            
            print(f"\n{'='*60}")
            print(f"验证因子 {i+1}")
            print('='*60)
            
            try:
                # 计算因子值
                factor_values = test_engine.evaluate_tree(ind)
                
                # 创建验证器并生成报告
                validator = FactorValidator(test_df, factor_values, f"Factor_{i+1}")
                report = validator.generate_full_report(n_groups=10)
                
                # 绘制图表（可选）
                # validator.plot_results(save_path=f"factor_{i+1}_validation.png")
                
            except Exception as e:
                print(f"❌ 因子{i+1}验证失败: {e}")
        
        print("\n" + "="*60)
        print("✅ 验证完成！")
        print("="*60)
        
    else:
        print("\n" + "="*60)
        print("❌ 因子挖掘失败")
        print("="*60)

📂 加载数据
📂 加载数据
原始数据: 24678行, 11列
检测到日期格式: 2024-01-02
✅ 使用格式: %Y-%m-%d
✅ 249天 × 100只股票 = 24578条记录

🚀 LLM增强遗传编程 - 优化版
训练集: 149天, 14689行
验证集: 50天, 4950行
测试集: 50天, 4939行

构建训练引擎...
计算金融基因 (Train)...
  [1/21] ✓ gap
  [2/21] ✓ vol_ratio
  [3/21] ✓ amplitude
  [4/21] ✓ upper_shadow
  [5/21] ✓ lower_shadow
  [6/21] ✓ momentum5
  [7/21] ✓ momentum10
  [8/21] ✓ momentum20
  [9/21] ✓ momentum60
  [10/21] ✓ rsi
  [11/21] ✓ vol_price
  [12/21] ✓ macd
  [13/21] ✓ bbands_width
  [14/21] ✓ bbands_position
  [15/21] ✓ atr
  [16/21] ✓ vol_ma_ratio
  [17/21] ✓ price_to_ma5
  [18/21] ✓ price_to_ma20
  [19/21] ✓ price_to_ma60
  [20/21] ✓ vol_breakout
  [21/21] ✓ price_breakout
✅ Train 引擎就绪: 149天, 14689条记录

构建验证引擎...
计算金融基因 (Valid)...
  [1/21] ✓ gap
  [2/21] ✓ vol_ratio
  [3/21] ✓ amplitude
  [4/21] ✓ upper_shadow
  [5/21] ✓ lower_shadow
  [6/21] ✓ momentum5
  [7/21] ✓ momentum10
  [8/21] ✓ momentum20
  [9/21] ✓ momentum60
  [10/21] ✓ rsi
  [11/21] ✓ vol_price
  [12/21] ✓ macd
  [13/21] ✓ bbands_width
  [14/